# 🫁 AI for Health — Sleep Apnea Detection
**SRIP 2026 | Health Sensing Assignment**

This notebook implements a complete pipeline for detecting breathing irregularities during sleep using 1D CNN classification on physiological signals (Nasal Airflow, Thoracic Movement, SpO₂).

---
## Pipeline Overview

| Step | Script | Description |
|------|--------|-------------|
| 1 | `setup_data.py` | Organize raw files into standard folder structure |
| 2 | `scripts/vis.py` | Visualize 8-hour signals and annotate events |
| 3 | `scripts/create_dataset.py` | Filter signals, window, and label |
| 4 | `models/cnn_model.py` | 1D CNN architecture |
| 5 | `scripts/train_model.py` | LOPO Cross-Validation training |

> **All logs are saved to the `logs/` directory for later inspection.**

---
## Section 0 — Environment Setup

Clone the repository, install dependencies, and mount the project path to Python's import system.

In [ ]:
# Clone repo and install requirements
!git clone https://github.com/dmist08/AI-For-Health.git
!pip install -q -r AI-For-Health/requirements.txt

import sys
sys.path.append('/content/AI-For-Health')

# Confirm GPU availability
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

---
## Section 1 — Data Setup

Upload your data ZIP file to Colab, extract it, then run `setup_data.py` to standardize folder and file naming across all 5 participants.

In [ ]:
import zipfile, os

# --- CHANGE THIS if your zip file has a different name ---
ZIP_PATH  = "/content/Data.zip"
DEST_PATH = "/content/AI-For-Health/internship/"

os.makedirs(DEST_PATH, exist_ok=True)
print(f"Extracting {ZIP_PATH} ...")
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall(DEST_PATH)
print("Extraction complete.")
print("Contents:", os.listdir(DEST_PATH))

In [ ]:
# Organize raw files into Data/AP01..AP05 with standardized names
import subprocess, sys

result = subprocess.run(
    [sys.executable, '/content/AI-For-Health/setup_data.py'],
    cwd='/content/AI-For-Health',
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("STDERR:", result.stderr)

---
## Section 2 — Signal Visualization

Generate a 4-panel PDF visualization for each participant:
- Nasal Airflow (32 Hz)
- Thoracic Movement (32 Hz)
- SpO₂ (4 Hz)
- Sleep Stages with event overlays

PDFs are saved to `Visualizations/`.

In [ ]:
import subprocess, sys

PARTICIPANTS = ['AP01', 'AP02', 'AP03', 'AP04', 'AP05']
DATA_DIR = '/content/AI-For-Health/Data'

for p in PARTICIPANTS:
    print(f"Generating visualization for {p} ...")
    result = subprocess.run(
        [sys.executable, '/content/AI-For-Health/scripts/vis.py',
         '-name', f'{DATA_DIR}/{p}'],
        cwd='/content/AI-For-Health',
        capture_output=True, text=True
    )
    print(result.stdout.strip() or f"{p} done.")
    if result.returncode != 0:
        print("ERROR:", result.stderr)

print("\nAll visualizations saved to Visualizations/")

---
## Section 3 — Dataset Creation

**Signal processing pipeline:**
1. **Bandpass filter** Nasal Airflow + Thoracic Movement at 0.17–0.40 Hz (human breathing range)
2. **Lowpass filter** SpO₂ at 1.0 Hz (removes noise, preserves desaturation trends)
3. **Upsample** SpO₂ from 4 Hz → 32 Hz to match other signals
4. **Stack** all 3 channels into shape `(3, 960)` per window
5. **Window** into 30-second segments with 50% overlap
6. **Label** windows: `0 = Normal`, `1 = Event` (Hypopnea or Apnea)

Output: `Dataset/breathing_dataset.csv` and `Dataset/breathing_dataset.pkl`

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, '/content/AI-For-Health/scripts/create_dataset.py',
     '-in_dir', '/content/AI-For-Health/Data',
     '-out_dir', '/content/AI-For-Health/Dataset'],
    cwd='/content/AI-For-Health',
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("ERROR:", result.stderr)

In [ ]:
# Inspect the resulting label distribution
import pickle
from collections import Counter

with open('/content/AI-For-Health/Dataset/breathing_dataset.pkl', 'rb') as f:
    dataset = pickle.load(f)

print(f"{'Participant':<14} {'Total':>8} {'Normal':>8} {'Event':>8}")
print("-" * 42)
for p, data in sorted(dataset.items()):
    c = Counter(data['y'].tolist())
    print(f"{p:<14} {len(data['y']):>8} {c.get(0,0):>8} {c.get(1,0):>8}")

---
## Section 4 — Model Architecture

A simple single-branch 1D CNN:
- **Input:** `(Batch, 3, 960)` — 3 channels × 30 seconds at 32 Hz
- **3 ConvBlocks:** Conv1D → BatchNorm → ReLU → MaxPool → Dropout
- **Global Average Pooling** collapses the time dimension
- **Linear head:** outputs 2 logits (Normal vs Event)

In [ ]:
import torch
from models.cnn_model import SimpleCNN, count_parameters

model = SimpleCNN(num_classes=2)
print(f"Trainable parameters: {count_parameters(model):,}")

# Verify shapes
dummy = torch.randn(4, 3, 960)
out = model(dummy)
print(f"Input shape:  {dummy.shape}")
print(f"Output shape: {out.shape}  (expected: [4, 2])")

---
## Section 5 — Training: Leave-One-Participant-Out CV

**LOPO evaluation strategy:**  
In each of the 5 folds, one participant is held out as the test set. The model is trained on the remaining 4 participants. This ensures the model is always evaluated on a **completely unseen individual** — the hardest possible generalization test.

**Key design choices:**
- Normalization statistics computed from **training participants only** (no data leakage)
- Class-weighted `CrossEntropyLoss` to handle Normal >> Event imbalance
- **Early stopping** on validation loss (patience = 15 epochs)
- Results (confusion matrices, metrics CSV) saved to `results/`
- **All logs written to `logs/`** for offline inspection

In [ ]:
import subprocess, sys

result = subprocess.run(
    [sys.executable, '/content/AI-For-Health/scripts/train_model.py',
     '-dataset_path', '/content/AI-For-Health/Dataset/breathing_dataset.pkl'],
    cwd='/content/AI-For-Health',
    capture_output=True, text=True
)
print(result.stdout)
if result.returncode != 0:
    print("ERROR:", result.stderr)

---
## Section 6 — Results

Load and display all evaluation outputs: per-fold metrics table and confusion matrices.

In [ ]:
import pandas as pd

df = pd.read_csv('/content/AI-For-Health/results/lopo_metrics.csv')
display(df.style.format({
    'Accuracy': '{:.3f}', 'Precision': '{:.3f}',
    'Recall':   '{:.3f}', 'F1':        '{:.3f}'
}).set_caption('LOPO Cross-Validation Results'))

print(f"\nMean F1 across folds: {df['F1'].mean():.3f}")
print(f"Mean Recall:          {df['Recall'].mean():.3f}")

In [ ]:
from IPython.display import display, Image
import os

RESULTS_DIR = '/content/AI-For-Health/results'

# Show all per-fold confusion matrices
cm_files = sorted([f for f in os.listdir(RESULTS_DIR) if f.startswith('cm_fold')])
for f in cm_files:
    print(f"\n{f}")
    display(Image(filename=f'{RESULTS_DIR}/{f}', width=400))

# Aggregate
print("\nAggregate Confusion Matrix (all folds combined)")
display(Image(filename=f'{RESULTS_DIR}/cm_aggregate.png', width=450))

---
## Section 7 — Export & Download

Collect all outputs (Dataset, Visualizations, Results, Logs) into a single ZIP file for download.

In [ ]:
import zipfile, os
from google.colab import files

OUTPUT_ZIP = '/content/AI_for_Health_Outputs.zip'
REPO_ROOT  = '/content/AI-For-Health'

FOLDERS = ['Dataset', 'Visualizations', 'results', 'logs']

with zipfile.ZipFile(OUTPUT_ZIP, 'w', zipfile.ZIP_DEFLATED) as zf:
    for folder in FOLDERS:
        folder_path = os.path.join(REPO_ROOT, folder)
        if not os.path.exists(folder_path):
            print(f"Warning: {folder} not found, skipping.")
            continue
        for root, _, fnames in os.walk(folder_path):
            for fname in fnames:
                fpath = os.path.join(root, fname)
                arc   = os.path.relpath(fpath, REPO_ROOT)
                zf.write(fpath, arc)

print(f"Archive created: {OUTPUT_ZIP}")
files.download(OUTPUT_ZIP)